# Silver Layer - Generic Metadata-Driven Processing

Generic Bronze-to-Silver notebook driven by `TABLE_ID` metadata.


In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
%run ../helpers/NB_silver_metadata

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import coalesce, col, expr, from_json, row_number, to_timestamp, unbase64
from pyspark.sql.window import Window
import uuid

dbutils.widgets.text("TABLE_ID", "orders")
dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("SILVER_SCHEMA", "silver")
dbutils.widgets.text("CHECKPOINT_ROOT", "/Volumes/workspace/default/mnt/checkpoints")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")
dbutils.widgets.text("ALERT_CHANNEL", "log")
dbutils.widgets.text("WEBHOOK_URL", "")
dbutils.widgets.text("RUN_BOOTSTRAP_HOOKS", "false")


def parse_bool(raw_value: str) -> bool:
    return raw_value.strip().lower() in {"1", "true", "yes", "y"}


CONFIG = {
    "table_id": dbutils.widgets.get("TABLE_ID"),
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "silver_schema": dbutils.widgets.get("SILVER_SCHEMA"),
    "checkpoint_root": dbutils.widgets.get("CHECKPOINT_ROOT"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
    "run_bootstrap_hooks": parse_bool(dbutils.widgets.get("RUN_BOOTSTRAP_HOOKS")),
}

table_config = get_silver_table_config(CONFIG["table_id"])
PIPELINE_RUN_ID = str(uuid.uuid4())

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_config['bronze_table']}"
silver_table_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.{table_config['silver_table']}"
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"
checkpoint_path = f"{CONFIG['checkpoint_root'].rstrip('/')}/{table_config['checkpoint_name']}"


In [ ]:
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])
expected_schema = get_schema_contract(table_config['silver_contract_key'])
ensure_table_from_contract(spark, silver_table_fqn, expected_schema)

if CONFIG['run_bootstrap_hooks']:
    run_silver_bootstrap_hook(
        spark,
        table_config,
        silver_table_fqn,
        CONFIG['catalog'],
        CONFIG['silver_schema'],
    )

actual_schema = get_existing_schema(spark, silver_table_fqn)
print(f"Validating schema for {silver_table_fqn} with policy {CONFIG['schema_policy']}")

try:
    is_valid, drift_result = validate_schema_with_policy(
        spark,
        expected_schema=expected_schema,
        actual_schema=actual_schema,
        table_name=silver_table_fqn,
        drift_table=drift_table_fqn,
        source_system="postgres_cdc",
        pipeline_run_id=PIPELINE_RUN_ID,
        policy=CONFIG['schema_policy'],
        alert_channel=CONFIG['alert_channel'],
        webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None,
    )
    if drift_result['has_drift']:
        print(f"Schema drift detected: {drift_result['severity']}")
    else:
        print("Schema validation passed")
except SchemaDriftException as exc:
    dbutils.notebook.exit(f"Schema drift blocked pipeline: {exc}")


In [ ]:
payload_schema = get_schema_contract(table_config['cdc_contract_key'])

bronze_stream = (
    spark.readStream
    .option("mergeSchema", "true")
    .table(bronze_table_fqn)
)

parsed = (
    bronze_stream
    .select(
        col("offset").alias("bronze_offset"),
        col("partition").alias("bronze_partition"),
        col("kafka_timestamp"),
        from_json(col("value"), payload_schema).alias("data")
    )
    .select("bronze_offset", "bronze_partition", "kafka_timestamp", "data.payload.*")
)


In [ ]:
def coalesce_columns(source_paths: list[str]):
    columns = [col(path) for path in source_paths]
    if len(columns) == 1:
        return columns[0]
    return coalesce(*columns)


def build_events_df(parsed_df, field_mappings: list[dict]):
    working_df = parsed_df
    raw_columns_to_drop = []
    final_columns = []

    for field_mapping in field_mappings:
        target = field_mapping['target']
        transform = field_mapping.get('transform', 'plain')
        raw_column_name = f"__raw_{target}"

        if transform == 'plain':
            working_df = working_df.withColumn(target, coalesce_columns(field_mapping['source_paths']))
        elif transform == 'epoch_micros_to_timestamp':
            working_df = working_df.withColumn(raw_column_name, coalesce_columns(field_mapping['source_paths']))
            working_df = working_df.withColumn(target, to_timestamp((col(raw_column_name) / 1000000).cast('double')))
            raw_columns_to_drop.append(raw_column_name)
        elif transform == 'decimal_from_debezium_struct':
            bytes_column_name = f"{raw_column_name}_bytes"
            int_column_name = f"{raw_column_name}_int"
            precision = field_mapping['precision']
            scale = field_mapping['scale']
            working_df = working_df.withColumn(raw_column_name, coalesce_columns(field_mapping['source_paths']))
            working_df = working_df.withColumn(bytes_column_name, unbase64(col(f"{raw_column_name}.value")))
            working_df = working_df.withColumn(int_column_name, expr(f"cast(conv(hex({bytes_column_name}), 16, 10) as double)"))
            working_df = working_df.withColumn(
                target,
                expr(f"cast({int_column_name} / pow(10, {raw_column_name}.scale) as decimal({precision},{scale}))")
            )
            raw_columns_to_drop.extend([raw_column_name, bytes_column_name, int_column_name])
        else:
            raise ValueError(f"Unsupported transform '{transform}' for field '{target}'")

        final_columns.append(target)

    working_df = working_df.withColumn('event_ts_ms', col('ts_ms'))
    working_df = working_df.withColumn('event_time', to_timestamp((col('ts_ms') / 1000).cast('double')))
    final_columns.extend(['op', 'event_time', 'event_ts_ms', 'bronze_offset', 'bronze_partition', 'kafka_timestamp'])

    if raw_columns_to_drop:
        working_df = working_df.drop(*raw_columns_to_drop)

    return working_df.select(*final_columns)


events = build_events_df(parsed, table_config['field_mappings'])


In [ ]:
def upsert_to_silver(batch_df, batch_id):
    if not batch_df.take(1):
        return

    order_columns = [col(column_name).desc() for column_name in table_config['dedupe_order_columns']]
    window = Window.partitionBy(*table_config['primary_keys']).orderBy(*order_columns)
    latest = (
        batch_df
        .withColumn('__rn', row_number().over(window))
        .filter(col('__rn') == 1)
        .drop('__rn')
    )

    existing_columns = get_existing_columns(spark, silver_table_fqn)
    update_set, insert_values = build_merge_clauses(
        existing_columns,
        table_config['merge_core_fields'],
        include_inserted_dt=True,
        include_updated_dt=True,
    )

    delta_table = DeltaTable.forName(spark, silver_table_fqn)
    execute_merge(spark, delta_table, latest, update_set, insert_values)


query = (
    events.writeStream
    .foreachBatch(upsert_to_silver)
    .option('checkpointLocation', checkpoint_path)
    .option('mergeSchema', 'true')
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"Silver processing completed for {CONFIG['table_id']}. Run ID: {PIPELINE_RUN_ID}")
